## Setup

In [1]:
import sys 
import os
import IPython


# Env. vars to check if the notebook is running on colab, kaggle etc. 
IS_COLAB = "google.colab" in sys.modules 
IS_KAGGLE = "kaggle_secrets" in sys.modules 
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    # Install the packages 
    %pip install -q -U tensorflow-addons
    %pip install -q -U transformers
    %pip install -q -U datasets
    print("You can safely ignore the package incompatibility errors.")
    # Mount the drive 
    from google.colab import drive 
    drive.mount("/drive")

# NOTE: This needs to be done for correct notebook behavior with the transformers 
# library 
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm 

import random 
import shutil 
# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Pytorch imports 
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

from typing import List, Tuple

# To plot pretty figures
import seaborn as sns
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

sns.set_theme()

In [3]:
# Project Paths
NOTEBOOK_NAME = "1.0-POC"
PROJECT_ROOT_DIR = "."

# --- Result dirs. 
OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR,"outputs",NOTEBOOK_NAME)
os.makedirs(OUTPUT_DIR,exist_ok=True)


LOGGING_DIR = os.path.join(OUTPUT_DIR,"logs")
os.makedirs(LOGGING_DIR,exist_ok=True)

In [4]:
# Setup logging for this notebook 
from LLMDialogue.utils.logging import configure_logger, create_file_handler, create_stream_handler
from LLMDialogue.utils.general import get_system_device
import logging

configure_logger(
    handlers=[
        create_stream_handler(logging_level=logging.WARNING), 
        create_file_handler(
            outdir=LOGGING_DIR, 
            logging_level=logging.DEBUG, 
            reset_outdir=False # Keep logs across runs. 
        )
    ]
)


In [5]:

GLOBAL_DEVICE = get_system_device()
# Shared env. vars. 
GLOBAL_SEED = 42 
RESET_OUTDIR = True
# Configuring env.
if  GLOBAL_SEED is not None:
    # to make this notebook's output stable across runs
    np.random.seed(GLOBAL_SEED) 
    torch.manual_seed(GLOBAL_SEED)
# NOTE: Should be mps for mac. 
GLOBAL_DEVICE

device(type='mps')

In [6]:
#Reset all but the logs dir in the output folder
from LLMDialogue.utils.general import remove_dir, get_subdirs_in_dir
if RESET_OUTDIR:
    subdirs = get_subdirs_in_dir(OUTPUT_DIR)
    for subdir in subdirs:
        if not subdir in ['logs']:
            remove_dir(subdir)


In [7]:
from LLMDialogue.pipelines import CPPipelineConfig, CPPipeline
from LLMDialogue.models.gpt2 import GPT2

In [8]:
gpt2 = GPT2(seed=GLOBAL_SEED)
gpt2.load()

In [9]:
# Create the pipeline config 
config = CPPipelineConfig(
    model = gpt2, 
    utterances=['this is is a text'],
    N = -1,
    context_buffer_size=512,   
)

In [10]:
# Now, init the pipeline 
pipe = CPPipeline(config)


In [11]:
# Run the pipeline 
pipe.run()

Processing turns: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]


## Datasets

### In Conversation Corpus (ICC) 

#### Variant - No Labels

This variant of the ICC does not include any conversation start or end or 
speaker labels. 

In [ ]:
from LLMDialogue.datasets import load
from LLMDialogue.datasets.ICC import objectify as ICCObjectify, ICCDatasetSplits

In [ ]:
dset = load(
    dset_name="ICC", 
    config_name="no_labels"
)
print(dset)

In [ ]:
dset_splits = ICCObjectify(dset)
dset_splits.split_names


In [ ]:
dset_splits.data_df_map['train']

In [ ]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


#### Variant - Special Labels

In [ ]:
dset = load(
    dset_name="ICC", 
    config_name="labels"
)
print(dset)

In [ ]:
dset_splits = ICCObjectify(dset)
dset_splits.split_names

In [ ]:
dset_splits.data_df_map['train']

In [ ]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


### Speaker Identity

#### Variant - No Labels

In [ ]:
from LLMDialogue.datasets.speakerIdentity import SpeakerIdentityDataSplits, objectify as SIObjectify

In [ ]:
dset = load(
    dset_name="speakerIdentity",
    config_name="labels"
)

In [ ]:
dset_splits = SIObjectify(dset)
dset_splits.split_names

In [ ]:
dset_splits.data_df_map['test']

In [ ]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


#### Labels

In [ ]:
dset = load(
    dset_name="speakerIdentity",
    config_name="labels"
)

In [ ]:
dset_splits = SIObjectify(dset)
dset_splits.split_names

In [ ]:
dset_splits.data_df_map['test']

In [ ]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)

### Data Modules

These are required to be able to train the model 

In [ ]:
from LLMDialogue.dm.ICCdm import ICCHFLMDM
from LLMDialogue.dm.HFdm import HFLanguageModelingDM
from transformers import AutoTokenizer
from LLMDialogue.dm.ICCdm import ICCSpokenLanguageModelingDM
from LLMDialogue.tokenizers import SpokenDialogueTokenizer

#### ICC Language Modeling Data Module (Compatible with HuggingFace)

In [ ]:
# Load the tokenizer here
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
# Load the dm and prepare it.
dm = ICCHFLMDM(
    dset_variant="labels", 
    tokenizer=tokenizer, 
    save_dir=OUTPUT_DIR
)
dm.prepare_data()
dm.setup('fit')

#### ICC Spoken Language Modeling Data Module (Compatible with PL)

In [ ]:
tokenizer = SpokenDialogueTokenizer()

In [ ]:
dm = ICCSpokenLanguageModelingDM(
    dset_variant="labels", 
    tokenizer=tokenizer,
    save_dir=OUTPUT_DIR
)
dm.prepare_data()
dm.setup('fit')

### Fine-tuning

#### GPT-2

The first model we want to funetune is GPT2, which is an instance of a 
language model from hugging face. 

In [ ]:
from LLMDialogue.models import GPT2

In [ ]:
# Instantiate and load the model 
gpt2 = GPT2(seed=GLOBAL_SEED)
gpt2.load()
gpt2.to(GLOBAL_DEVICE)
print(gpt2)

In [ ]:
# Now, finetune the model using the appropriate data module 
# Load the dm and prepare it.
dm = ICCHFLMDM(
    dset_variant="labels", 
    tokenizer=gpt2.tokenizer, 
    save_dir=OUTPUT_DIR
)


In [ ]:
# Now, finetune the model. 
# gpt2.finetune(
#     dm=dm, 
#     save_dir=f"{OUTPUT_DIR}/models/gpt2",
#     num_train_epochs=1
# )

### TurnGPT

In [ ]:
from LLMDialogue.models import TurnGPT 


In [ ]:
turngpt = TurnGPT(seed=GLOBAL_SEED)
turngpt.load(model_head="LMHead")
turngpt.to(device=GLOBAL_DEVICE)
print(turngpt)

In [ ]:
# Load the appropriate dm and finetune
dm = ICCSpokenLanguageModelingDM(
    dset_variant="labels", 
    tokenizer=tokenizer,
    save_dir=OUTPUT_DIR
)

In [ ]:
# turngpt.finetune(
#     dm=dm,
#     save_dir=f"{OUTPUT_DIR}/models/turngpt",
#     max_epochs=1
# )

### Pipelines

In [ ]:
from LLMDialogue.pipelines import CPPipelineConfig, CPPipeline

In [ ]:
gpt2 = GPT2(seed=GLOBAL_SEED)
gpt2.load()

In [ ]:
# Create the pipeline config 
config = CPPipelineConfig(
    model = gpt2, 
    utterances=['this is is a text'],
    N = -1,
    context_buffer_size=512,   
)

In [ ]:
# Now, init the pipeline 
pipe = CPPipeline(config)


